# Imports

In [1]:
import os
import urllib.request
import zipfile
import pandas as pd


# Local Directory Setup

In [2]:

# Local Directory Setup 
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Local Data Directory set to: {DATA_DIR}")

# CRITICAL: GitHub Security (.gitignore)
gitignore_path = os.path.join(os.getcwd(), '.gitignore')
with open(gitignore_path, 'a+') as f:
    f.seek(0)
    content = f.read()
    if 'TFE_Data/' not in content:
        f.write("\n# Ignore Dataset Folder\nTFE_Data/\n")
        f.write("__pycache__/\n*.npy\n*.zip\n")
print(".gitignore security is active.")


Local Data Directory set to: /home/aysel/tfe/TFE_Data
.gitignore security is active.


In [4]:
def download_and_extract(url, extract_to): 
    """Downloads a zip file from the specified URL and extracts it to the given directory."""
    
    os.makedirs(extract_to, exist_ok=True)
    zip_path = os.path.join(extract_to, "temp.zip")
    
    if not os.listdir(extract_to): 
        print(f"Downloading from {url}...")
        urllib.request.urlretrieve(url, zip_path) # Download the zip file
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to) # Extract the contents
        os.remove(zip_path) # Clean up the zip file
        print("Extraction complete.")
    else:
        print(f"Files already exist in {extract_to}. Skipping download.")

# Flikr8k

In [7]:
def build_flickr8k():
    print("\n--- BUILDING FLICKR8K ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr8k")
    img_dir = os.path.join(dataset_dir, "Images")
    txt_dir = os.path.join(dataset_dir, "Text")
    
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip", img_dir)
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip", txt_dir)
    
    actual_images_dir = os.path.join(img_dir, "Flicker8k_Dataset")
    token_path = os.path.join(txt_dir, "Flickr8k.token.txt")
    data = []

    print("Verifying physical files and syncing captions...")
    with open(token_path, 'r') as f:
        for line in f.read().splitlines():
            if not line: continue
            parts = line.split('\t')
            if len(parts) == 2:
                img_name = parts[0].split('#')[0]
                full_path = os.path.join(actual_images_dir, img_name)
                
                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name, 
                        "image_path": full_path, 
                        "caption": parts[1].strip()
                    })

    df_flat = pd.DataFrame(data) # Create a flat DataFrame with one row per caption
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index() # Group captions by image
    df.rename(columns={'caption': 'captions'}, inplace=True) # Rename column to 'captions' for clarity
    
    save_path = os.path.join(DATA_DIR, 'df_Flickr8k.pkl') # Save the DataFrame as a pickle file
    df.to_pickle(save_path) 
    
    print(f"Flickr8k Ready: {len(df)} images secured.")
    return "Flickr8k"

In [6]:
def build_flickr8k_subset(subset_size=2000):
    print("\n--- BUILDING FLICKR8K SUBSET ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr8k")
    img_dir = os.path.join(dataset_dir, "Images")
    txt_dir = os.path.join(dataset_dir, "Text")
    
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip", img_dir)
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip", txt_dir)
    
    actual_images_dir = os.path.join(img_dir, "Flicker8k_Dataset")
    token_path = os.path.join(txt_dir, "Flickr8k.token.txt")
    data = []

    print("Verifying physical files and syncing captions...")
    with open(token_path, 'r') as f:
        for line in f.read().splitlines():
            if not line: continue
            parts = line.split('\t')
            if len(parts) == 2:
                img_name = parts[0].split('#')[0]
                full_path = os.path.join(actual_images_dir, img_name)
                
                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name, 
                        "image_path": full_path, 
                        "caption": parts[1].strip()
                    })

    df_flat = pd.DataFrame(data) # Create a flat DataFrame with one row per caption
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index() # Group captions by image
    df.rename(columns={'caption': 'captions'}, inplace=True) # Rename column to 'captions' for clarity
    
    subset_df = df.head(subset_size).copy() # Select a subset of the data
    save_path = os.path.join(DATA_DIR, 'subset_df_Flickr8k.pkl') # Save the subset DataFrame as a pickle file
    subset_df.to_pickle(save_path) 
    
    print(f" Flickr8k Subset Ready: {len(subset_df)} images secured.")
    return "Flickr8k Subset"

# Execution of Downloading Flikr8k

In [8]:
if __name__ == "__main__":
    ds_name = build_flickr8k_subset(subset_size=2000)
    ds_name = build_flickr8k()



--- BUILDING FLICKR8K SUBSET ---
Files already exist in /home/aysel/tfe/TFE_Data/Flickr8k/Images. Skipping download.
Files already exist in /home/aysel/tfe/TFE_Data/Flickr8k/Text. Skipping download.
Verifying physical files and syncing captions...
 Flickr8k Subset Ready: 2000 images secured.

--- BUILDING FLICKR8K ---
Files already exist in /home/aysel/tfe/TFE_Data/Flickr8k/Images. Skipping download.
Files already exist in /home/aysel/tfe/TFE_Data/Flickr8k/Text. Skipping download.
Verifying physical files and syncing captions...
Flickr8k Ready: 8091 images secured.
